# ClusterAPI

Cluster API ist ein Kubernetes-Projekt zur deklarativen Erstellung und Verwaltung von Kubernetes-Clustern. Es bildet den gesamten Cluster-Lifecycle über Kubernetes-Ressourcen und Controller ab.

Die Ressourcen sind hierarchisch aufgebaut: Ein `Cluster` beschreibt den logischen Kubernetes-Cluster. Die Control Plane wird über die Ressource `KubeadmControlPlane` verwaltet, während Worker Nodes über `MachineDeployments`, darunterliegende `MachineSets` und einzelne `Machines` abgebildet werden. Jede `Machine` repräsentiert dabei eine konkrete virtuelle oder physische Instanz, die später als Kubernetes Node registriert wird.

```text
Cluster
├── ControlPlane
│   └── Machines
└── MachineDeployments
    └── MachineSets
        └── Machines
            └── Kubernetes Nodes
```

Im Gegensatz zu Terraform prüft Cluster API laufend, ob der tatsächliche Zustand dem gewünschten Zustand entspricht, und korrigiert Abweichungen automatisch. Es ermöglicht einheitliche Cluster-Vorlagen, kontrollierte Updates, das Skalieren von Nodes und den automatischen Ersatz defekter Maschinen.

Da Cluster, Control Plane und Worker Nodes als Kubernetes-Ressourcen verwaltet werden, lassen sie sich direkt mit GitOps, Berechtigungen, Richtlinien und Monitoring verbinden. 

**Der grösste Vorteil liegt deshalb nicht nur in der Erstellung von Clustern, sondern vor allem in deren einheitlichem und automatisiertem Betrieb.**

---

Installation CLI

In [ ]:
%%bash
curl -L https://github.com/kubernetes-sigs/cluster-api/releases/download/v1.13.3/clusterctl-linux-amd64 -o clusterctl
sudo install -o root -g root -m 0755 clusterctl /usr/local/bin/clusterctl

### Load Balancer

Im Zusammenhang mit Cluster API wird ein Load Balancer benötigt, damit die automatisch erstellten Control-Plane-Nodes über eine gemeinsame, feste Adresse erreichbar sind. 

Cluster API kann dadurch Control-Plane-Nodes austauschen oder erweitern, ohne dass sich der Zugriffspunkt auf den Kubernetes-API-Server ändert.

In [ ]:
%%bash
METALLB_VER=$(curl "https://api.github.com/repos/metallb/metallb/releases/latest" | jq -r ".tag_name")
kubectl apply -f "https://raw.githubusercontent.com/metallb/metallb/${METALLB_VER}/config/manifests/metallb-native.yaml"
kubectl wait pods -n metallb-system -l app=metallb,component=controller --for=condition=Ready --timeout=10m
kubectl wait pods -n metallb-system -l app=metallb,component=speaker --for=condition=Ready --timeout=2m
cat <<EOF | kubectl apply -f -
apiVersion: metallb.io/v1beta1
kind: IPAddressPool
metadata:
  name: capi-ip-pool
  namespace: metallb-system
spec:
  addresses:
  - 10.10.0.30-10.10.0.90
---
apiVersion: metallb.io/v1beta1
kind: L2Advertisement
metadata:
  name: empty
  namespace: metallb-system
EOF


### ClusterAPI Initialisieren

Initialisieren des Management-Cluster mit dem KubeVirt-Provider. 

Dabei werden die benötigten Cluster-API-Komponenten sowie die Controller und Custom Resources für KubeVirt installiert. Der Management-Cluster kann anschliessend KubeVirt-basierte Workload-Cluster erstellen, verwalten, skalieren und aktualisieren.


In [ ]:
%%bash
clusterctl init --infrastructure kubevirt

---
### Headlamp

Das Headlamp ist eine moderne alternative Weboberfläche für Kubernetes. Im Gegensatz zum klassischen Kubernetes Dashboard fokussiert sich Headlamp stärker auf Übersichtlichkeit, Multi-Cluster-Unterstützung und eine benutzerfreundliche Navigation.


In [ ]:
%%bash
echo "HeadLamp: http://"$(cat ~/data/server-ip)":30444"
kubectl create token headlamp-admin -n kube-system --duration=48h

### Cluster-Konfiguration erzeugen

In diesem Schritt wird festgelegt, welches virtuelle Maschinen-Image für die Cluster-Nodes verwendet wird und welche Kubernetes-Version installiert werden soll. Danach erzeugt Cluster API die vollständige Konfiguration für einen neuen Workload-Cluster mit einem Control-Plane-Node, zwei Worker Nodes und einem Load Balancer.

* [siehe auch Cluster Images](https://quay.io/repository/capk/ubuntu-2404-container-disk?tab=tags&tag=latest)

In [ ]:
%%bash
export NODE_VM_IMAGE_TEMPLATE="quay.io/capk/ubuntu-2404-container-disk:v1.33.5"
export CAPK_GUEST_K8S_VERSION="${NODE_VM_IMAGE_TEMPLATE/*:/}"
export CRI_PATH="unix:///var/run/containerd/containerd.sock"
    
clusterctl generate cluster capi-kubevirt \
  --infrastructure="kubevirt" \
  --flavor lb \
  --kubernetes-version ${CAPK_GUEST_K8S_VERSION} \
  --control-plane-machine-count=1 \
  --worker-machine-count=2 \
  --target-namespace=capi-kubevirt \
  > capi-kubevirt.yaml


Die erzeugte Datei `capi-kubevirt.yaml` enthält alle Ressourcen, die Cluster API benötigt, um den Cluster auf der KubeVirt-Umgebung bereitzustellen.

In [ ]:
%%bash
kubectl create namespace capi-kubevirt
kubectl apply -f capi-kubevirt.yaml

### Workaround für den fehlenden Containerd-Pfad bei den Worker Nodes

Im generierten Worker-Template ist der CRI-Socket für Containerd nicht explizit definiert. Dadurch kann kubeadm beim Beitritt eines Worker Nodes unter Umständen nicht erkennen, welche Container-Runtime verwendet werden soll.

Mit der folgenden Anpassung wird der Containerd-Socket im KubeadmConfigTemplate fest eingetragen. Neue Worker Nodes verwenden damit beim Cluster-Beitritt den korrekten CRI-Endpunkt.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: bootstrap.cluster.x-k8s.io/v1beta1
kind: KubeadmConfigTemplate
metadata:
  name: capi-kubevirt-md-0
  namespace: capi-kubevirt
spec:
  template:
    spec:
      joinConfiguration:
        nodeRegistration:
          criSocket: unix:///var/run/containerd/containerd.sock
          kubeletExtraArgs: {}
EOF

### Status der erstellten Ressourcen prüfen

Nach dem Erstellen des Workload-Clusters können die wichtigsten Cluster-API- und KubeVirt-Ressourcen angezeigt werden:

Dabei ist insbesondere der Status der `KubeadmControlPlane` relevant. Die Control Plane muss vollständig bereit sein, bevor auf den neuen Workload-Cluster zugegriffen werden kann. Der Aufbau kann einige Zeit dauern, da Cluster API zuerst die virtuellen Maschinen erstellt, Kubernetes initialisiert und die Control-Plane-Nodes registriert.


In [ ]:
%%bash
kubectl -n capi-kubevirt get machinedeployment,machineset,machine,vmi

Sobald die `KubeadmControlPlane` bereit ist, kann die Kubeconfig des Workload-Clusters erzeugt werden:

Anschliessend kann mit dieser Kubeconfig auf den neu erstellten Cluster zugegriffen und geprüft werden, ob alle Systemkomponenten erfolgreich gestartet wurden:

In [ ]:
%%bash
clusterctl get kubeconfig --namespace capi-kubevirt capi-kubevirt > capi-kubevirt.kubeconfig
kubectl --kubeconfig=$HOME/data/.kube/capi-kubevirt get pods -A

### Flannel als Overlay Netzwerk installieren


In [ ]:
%%bash
kubectl --kubeconfig=$HOME/data/.kube/capi-kubevirt apply -f - <<EOF
$(curl -fsSL \
  https://github.com/flannel-io/flannel/releases/download/v0.28.7/kube-flannel.yml |
  sed 's|10.244.0.0/16|10.243.0.0/16|g')
EOF

In [ ]:
%%bash
kubectl -n capi-kubevirt get machinedeployment,machineset,machine,vmi,services
export KUBECONFIG=$HOME/data/.kube/capi-kubevirt
kubectl get nodes
kubectl get pods,services -o wide -A

### Headlamp starten

Anschliessend wird Headlamp gestartet, um den Workload-Cluster über eine grafische Oberfläche zu überwachen und Kubernetes-Ressourcen einfacher zu verwalten.

Die IP-Adresse des Clusters ist ggf. anzupassen.

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kubevirt
kubectl apply -f https://raw.githubusercontent.com/headlamp-k8s/headlamp/main/kubernetes-headlamp.yaml
kubectl -n kube-system create serviceaccount headlamp-admin
kubectl create clusterrolebinding headlamp-admin --serviceaccount=kube-system:headlamp-admin --clusterrole=cluster-admin

kubectl patch svc headlamp \
  -n kube-system \
  --type='merge' \
  -p '{
    "spec": {
      "type": "NodePort",
      "ports": [
        {
          "port": 80,
          "targetPort": 4466,
          "nodePort": 30444
        }
      ]
    }
  }'
echo "HeadLamp: http://"10.10.0.31":30444"
kubectl create token headlamp-admin -n kube-system --duration=48h

**Portweiterleitung (Port Forwarding)**

Ein wichtiger Aspekt bei der Arbeit mit Kubernetes ist die Fähigkeit, auf die im Cluster ausgeführten Dienste von aussen zuzugreifen. 

Mit einer einfach Portweiterleitung (Port Forwarding) können wir auf Headlamp zugreifen.

Mittels des **Stop Button** (oben) können wir die Weiterleitung stoppen.

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/capi-kubevirt
echo http://"$(cat ~/data/server-ip)":8888
kubectl port-forward --namespace kube-system --address 0.0.0.0 service/headlamp 8888:80

---

### Skalierung von Worker Nodes

Worker Nodes können über Machines hinzugefügt oder entfernt werden, um die Rechenkapazität des Clusters anzupassen. Die Skalierung erfolgt in der Regel über MachineSets oder MachineDeployments.

Beim Entfernen einer Machine wird der zugehörige Node kontrolliert geleert und anschliessend zusammen mit seiner Infrastruktur gelöscht.

**Links**
* [Skalierung](https://cluster-api.sigs.k8s.io/tasks/automated-machine-management/scaling)
* [Auto Skalierung](https://cluster-api.sigs.k8s.io/tasks/automated-machine-management/autoscaling)


In [ ]:
%%bash
kubectl get --namespace capi-kubevirt machinedeployment 

In [ ]:
%%bash
kubectl scale machinedeployment capi-kubevirt-md-0 --replicas=3

In [ ]:
%%bash
kubectl get --namespace capi-kubevirt machinedeployment 
kubectl get --namespace capi-kubevirt cluster,kubeadmcontrolplane,vmi,services
export KUBECONFIG=$HOME/data/.kube/capi-kubevirt
kubectl get nodes

- - -

### Updates von Kubernetes-Clustern

Cluster API ermöglicht kontrollierte Updates von Kubernetes-Clustern. Dabei wird zuerst die Control Plane und danach werden die Worker Nodes schrittweise auf die neue Version aktualisiert, damit der Cluster während des Updates möglichst verfügbar bleibt. ([cluster-api.sigs.k8s.io][1])

[1]: https://cluster-api.sigs.k8s.io/tasks/upgrading-clusters "Upgrading management and workload clusters - The Cluster API Book"
* [für ein nicht ClusterClass-basierter Cluster basierender Cluster](https://cluster-api.sigs.k8s.io/tasks/upgrading-clusters)

Neue Templates erstellen


In [ ]:
%%bash
export NAMESPACE="capi-kubevirt"
export CLUSTER_NAME="capi-kubevirt"
export NEW_VERSION="v1.34.1"
export NEW_IMAGE="quay.io/capk/ubuntu-2404-container-disk:${NEW_VERSION}"

export KCP_NAME="$(
  kubectl -n "${NAMESPACE}" get kubeadmcontrolplane \
    -l cluster.x-k8s.io/cluster-name="${CLUSTER_NAME}" \
    -o jsonpath='{.items[0].metadata.name}'
)"

export MD_NAME="$(
  kubectl -n "${NAMESPACE}" get machinedeployment \
    -l cluster.x-k8s.io/cluster-name="${CLUSTER_NAME}" \
    -o jsonpath='{.items[0].metadata.name}'
)"

export OLD_CP_TEMPLATE="$(
  kubectl -n "${NAMESPACE}" get kubeadmcontrolplane "${KCP_NAME}" \
    -o jsonpath='{.spec.machineTemplate.spec.infrastructureRef.name}'
)"

export OLD_WORKER_TEMPLATE="$(
  kubectl -n "${NAMESPACE}" get machinedeployment "${MD_NAME}" \
    -o jsonpath='{.spec.template.spec.infrastructureRef.name}'
)"

export VERSION_SUFFIX="${NEW_VERSION//./-}"
export NEW_CP_TEMPLATE="${CLUSTER_NAME}-control-plane-${VERSION_SUFFIX}"
export NEW_WORKER_TEMPLATE="${CLUSTER_NAME}-md-0-${VERSION_SUFFIX}"

printf 'KCP:              %s\n' "${KCP_NAME}"
printf 'MachineDeployment: %s\n' "${MD_NAME}"
printf 'Control Plane:     %s -> %s\n' "${OLD_CP_TEMPLATE}" "${NEW_CP_TEMPLATE}"
printf 'Worker:            %s -> %s\n' "${OLD_WORKER_TEMPLATE}" "${NEW_WORKER_TEMPLATE}"

kubectl -n "${NAMESPACE}" get kubevirtmachinetemplate "${OLD_CP_TEMPLATE}" -o json |
jq \
  --arg name "${NEW_CP_TEMPLATE}" \
  --arg image "${NEW_IMAGE}" '
    del(
      .metadata.uid,
      .metadata.resourceVersion,
      .metadata.creationTimestamp,
      .metadata.generation,
      .metadata.managedFields,
      .metadata.ownerReferences,
      .status
    )
    | .metadata.name = $name
    | walk(
        if type == "object"
           and has("image")
           and (.image | type) == "string"
           and (.image | contains("container-disk"))
        then .image = $image
        else .
        end
      )
  ' |
kubectl apply -f -

kubectl -n "${NAMESPACE}" get kubevirtmachinetemplate "${OLD_WORKER_TEMPLATE}" -o json |
jq \
  --arg name "${NEW_WORKER_TEMPLATE}" \
  --arg image "${NEW_IMAGE}" '
    del(
      .metadata.uid,
      .metadata.resourceVersion,
      .metadata.creationTimestamp,
      .metadata.generation,
      .metadata.managedFields,
      .metadata.ownerReferences,
      .status
    )
    | .metadata.name = $name
    | walk(
        if type == "object"
           and has("image")
           and (.image | type) == "string"
           and (.image | contains("container-disk"))
        then .image = $image
        else .
        end
      )
  ' |
kubectl apply -f -    


**Control Plane updaten**

Muss vor den Worker Nodes erfolgen

In [ ]:
%%bash
export NAMESPACE="capi-kubevirt"
export CLUSTER_NAME="capi-kubevirt"
export NEW_VERSION="v1.34.1"
export VERSION_SUFFIX="${NEW_VERSION//./-}"
export NEW_CP_TEMPLATE="${CLUSTER_NAME}-control-plane-${VERSION_SUFFIX}"
export KCP_NAME="$(
  kubectl -n "${NAMESPACE}" get kubeadmcontrolplane \
    -l cluster.x-k8s.io/cluster-name="${CLUSTER_NAME}" \
    -o jsonpath='{.items[0].metadata.name}'
)"

printf 'Control Plane:     %s\n' "${NEW_CP_TEMPLATE}"

kubectl -n "${NAMESPACE}" patch kubeadmcontrolplane "${KCP_NAME}" \
  --type=json \
  -p="[
    {
      \"op\": \"replace\",
      \"path\": \"/spec/version\",
      \"value\": \"${NEW_VERSION}\"
    },
    {
      \"op\": \"replace\",
      \"path\": \"/spec/machineTemplate/spec/infrastructureRef/name\",
      \"value\": \"${NEW_CP_TEMPLATE}\"
    }
  ]"

kubectl -n capi-kubevirt patch kubeadmcontrolplane \
  capi-kubevirt-control-plane \
  --type=json \
  -p='[
    {
      "op": "replace",
      "path": "/spec/machineTemplate/spec/infrastructureRef/kind",
      "value": "KubevirtMachineTemplate"
    }
  ]'

In [ ]:
%%bash
kubectl -n capi-kubevirt get machinedeployment,machineset,machine,vmi
export KUBECONFIG=$HOME/data/.kube/capi-kubevirt
kubectl get nodes

**Worker Nodes updaten**

Nachdem die Control Plane ausgetauscht wurde, können die Worker upgedatet werden.

In [ ]:
%%bash
export NAMESPACE="capi-kubevirt"
export CLUSTER_NAME="capi-kubevirt"
export NEW_VERSION="v1.34.1"
export VERSION_SUFFIX="${NEW_VERSION//./-}"
export NEW_WORKER_TEMPLATE="${CLUSTER_NAME}-md-0-${VERSION_SUFFIX}"
export KCP_NAME="$(
  kubectl -n "${NAMESPACE}" get kubeadmcontrolplane \
    -l cluster.x-k8s.io/cluster-name="${CLUSTER_NAME}" \
    -o jsonpath='{.items[0].metadata.name}'
)"
export NEW_WORKER_TEMPLATE="${CLUSTER_NAME}-md-0-${VERSION_SUFFIX}"
printf 'Worker:            %s\n' "${NEW_WORKER_TEMPLATE}"

kubectl -n capi-kubevirt patch machinedeployment \
  capi-kubevirt-md-0 \
  --type=json \
  -p='[
    {
      "op": "replace",
      "path": "/spec/template/spec/version",
      "value": "v1.34.1"
    },
    {
      "op": "replace",
      "path": "/spec/template/spec/infrastructureRef/apiGroup",
      "value": "infrastructure.cluster.x-k8s.io"
    },
    {
      "op": "replace",
      "path": "/spec/template/spec/infrastructureRef/kind",
      "value": "KubevirtMachineTemplate"
    },
    {
      "op": "replace",
      "path": "/spec/template/spec/infrastructureRef/name",
      "value": "capi-kubevirt-md-0-v1-34-1"
    }
  ]'


In [ ]:
%%bash
kubectl -n capi-kubevirt get machinedeployment,machineset,machine,vmi
export KUBECONFIG=$HOME/data/.kube/capi-kubevirt
kubectl get nodes

---

### Aufräumen


In [ ]:
%%bash
kubectl delete --namespace capi-kubevirt cluster capi-kubevirt
kubectl delete namespace capi-kubevirt